<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


---

<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a> 所著《<a href="http://mng.bz/orYv">从零构建大语言模型</a>》一书的补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Understanding the Difference Between Embedding Layers and Linear Layers


---

# 理解嵌入层与线性层的区别


- Embedding layers in PyTorch accomplish the same as linear layers that perform matrix multiplications; the reason we use embedding layers is computational efficiency
- We will take a look at this relationship step by step using code examples in PyTorch


---

- PyTorch中的嵌入层（Embedding layers）与执行矩阵乘法的线性层（linear layers）功能相同；我们使用嵌入层的原因是计算效率
- 我们将通过PyTorch代码示例逐步探讨这种关系


In [1]:
import torch

print("PyTorch version:", torch.__version__)

PyTorch version: 2.3.1


<br>
&nbsp;

## Using nn.Embedding


---

<br>
&nbsp;

## 使用 nn.Embedding


In [2]:
# Suppose we have the following 3 training examples,
# which may represent token IDs in a LLM context
idx = torch.tensor([2, 3, 1])

# The number of rows in the embedding matrix can be determined
# by obtaining the largest token ID + 1.
# If the highest token ID is 3, then we want 4 rows, for the possible
# token IDs 0, 1, 2, 3
num_idx = max(idx)+1

# The desired embedding dimension is a hyperparameter
out_dim = 5

- Let's implement a simple embedding layer:


---

- 让我们来实现一个简单的嵌入层：


In [3]:
# We use the random seed for reproducibility since
# weights in the embedding layer are initialized with
# small random values
torch.manual_seed(123)

embedding = torch.nn.Embedding(num_idx, out_dim)

We can optionally take a look at the embedding weights:


---

我们可以选择性地查看嵌入权重：


In [4]:
embedding.weight

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  1.5810],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015],
        [ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953]], requires_grad=True)

- We can then use the embedding layers to obtain the vector representation of a training example with ID 1:


---

- 然后我们可以使用嵌入层来获取ID为1的训练样本的向量表示：


In [5]:
embedding(torch.tensor([1]))

tensor([[ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

- Below is a visualization of what happens under the hood:


---

以下是底层运作的可视化展示：


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/1.png" width="400px">


---

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/1.png" width="400px" alt="嵌入与线性层">


- Similarly, we can use embedding layers to obtain the vector representation of a training example with ID 2:


---

- 同样，我们可以使用嵌入层来获取 ID 为 2 的训练样本的向量表示：


In [6]:
embedding(torch.tensor([2]))

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315]],
       grad_fn=<EmbeddingBackward0>)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/2.png" width="400px">


---

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/2.png" width="400px">


- Now, let's convert all the training examples we have defined previously:


---

现在，让我们转换之前定义的所有训练样本：


In [7]:
idx = torch.tensor([2, 3, 1])
embedding(idx)

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

- Under the hood, it's still the same look-up concept:


---

- 底层仍然是相同的查找概念：


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/3.png" width="450px">


---

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/3.png" width="450px">


<br>
&nbsp;

## Using nn.Linear


---

<br>
&nbsp;

## 使用 nn.Linear


- Now, we will demonstrate that the embedding layer above accomplishes exactly the same as `nn.Linear` layer on a one-hot encoded representation in PyTorch
- First, let's convert the token IDs into a one-hot representation:


---

- 现在，我们将展示上述嵌入层在 PyTorch 中对独热编码表示执行的操作，与 `nn.Linear` 层完全相同
- 首先，让我们将 token ID 转换为独热表示：


In [8]:
onehot = torch.nn.functional.one_hot(idx)
onehot

tensor([[0, 0, 1, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]])

- Next, we initialize a `Linear` layer, which carries out a matrix multiplication $X W^\top$:


---

- 接下来，我们初始化一个 `Linear` 层，该层执行矩阵乘法 $X W^\top$：


In [9]:
torch.manual_seed(123)
linear = torch.nn.Linear(num_idx, out_dim, bias=False)
linear.weight

Parameter containing:
tensor([[-0.2039,  0.0166, -0.2483,  0.1886],
        [-0.4260,  0.3665, -0.3634, -0.3975],
        [-0.3159,  0.2264, -0.1847,  0.1871],
        [-0.4244, -0.3034, -0.1836, -0.0983],
        [-0.3814,  0.3274, -0.1179,  0.1605]], requires_grad=True)

- Note that the linear layer in PyTorch is also initialized with small random weights; to directly compare it to the `Embedding` layer above, we have to use the same small random weights, which is why we reassign them here:


---

- 请注意，PyTorch 中的线性层也是用较小的随机权重初始化的；为了直接与上面的 `Embedding` 层进行比较，我们必须使用相同的较小随机权重，这就是为什么我们在此处重新赋值：


In [10]:
linear.weight = torch.nn.Parameter(embedding.weight.T)

- Now we can use the linear layer on the one-hot encoded representation of the inputs:


---

- 现在我们可以在输入的独热编码表示上使用线性层：


In [11]:
linear(onehot.float())

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]], grad_fn=<MmBackward0>)

As we can see, this is exactly the same as what we got when we used the embedding layer:


---

正如我们所见，这与我们使用embedding层时得到的结果完全相同：


In [12]:
embedding(idx)

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

- What happens under the hood is the following computation for the first training example's token ID:


---

- 底层运作如下，针对第一个训练样本的 token ID 进行以下计算：


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/4.png" width="450px">


---

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/4.png" width="450px">


- And for the second training example's token ID:


---

- 第二个训练样本的token ID：


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/5.png" width="450px">


---

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/5.png" width="450px">


- Since all but one index in each one-hot encoded row are 0 (by design), this matrix multiplication is essentially the same as a look-up of the one-hot elements
- This use of the matrix multiplication on one-hot encodings is equivalent to the embedding layer look-up but can be inefficient if we work with large embedding matrices, because there are a lot of wasteful multiplications by zero


---

- 由于每个独热编码行中除一个索引外其余全为0（设计如此），这种矩阵乘法本质上等同于查找独热编码元素
- 在独热编码上使用矩阵乘法等效于嵌入层查找，但当处理大型嵌入矩阵时可能效率低下，因为存在大量与零相乘的冗余计算
